# Phase 2A — nnUNet v2 Fine-Tuning (BraTS 2024 Post-Treatment)
## 3 output channels: WT / TC / ET (Correct BraTS 2024 labels: ET=3, RC=4 excluded)

**Pretrained**: nnUNet BraTS 2021 fold_3 (WT=0.9005, TC=0.8673, ET=0.8509)

Fine-tuning on BraTS 2024 Post-Treatment with **correct BraTS 2024 label mapping**:
- **NETC** (label 1) = Non-Enhancing Tumor Core
- **SNFH** (label 2) = Surrounding Non-enhancing FLAIR Hyperintensity
- **ET** (label 3) = Enhancing Tissue ← active tumor
- **RC** (label 4) = Resection Cavity → **excluded** from all regions

**Output regions**: WT=1+2+3 | TC=1+3 | ET=3

**Phase 2 constraints applied:**
- `safe_loader_iter` for corrupt NIfTI files
- `CacheDataset(cache_rate=0.05)` for Train, plain `Dataset` for Val
- No `PersistentDataset` (kills Kaggle disk)
- `.nii_gz` symlink trick for Kaggle auto-extraction prevention

In [ ]:
import re
import math
import time
import json
import random
import shutil
import subprocess
import numpy as np
import torch
import torch.nn.functional as F
import gc
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'nnunet'
PATCH       = [128, 128, 128]
# 3-channel output matching BraTS 2021 pretrained checkpoint (no RC)
# Output order: WT, TC, ET  (same region-based sigmoid as nnUNet)
REGIONS     = ['WT', 'TC', 'ET']
OUTPUT_ROOT = Path('/kaggle/working/phase2_nnunet')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Model: {MODEL_NAME} | Patch: {PATCH} | Regions: {REGIONS}')


In [ ]:
import subprocess, sys, json, time, math, os, shutil
import numpy as np
import torch
import torch.nn.functional as F

# Install nnunetv2 (needed to reconstruct exact architecture from plans)
try:
    import nnunetv2
    try:
        ver = nnunetv2.__version__
    except AttributeError:
        import importlib.metadata
        try: ver = importlib.metadata.version('nnunetv2')
        except Exception: ver = 'installed'
    print(f'nnunetv2 {ver} ready')
except ImportError:
    print('Installing nnunetv2 ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nnunetv2', '-q'])
    import nnunetv2
    print('nnunetv2 installed')

try:
    import monai
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai[all]', '-q'])
    import monai

import monai.transforms as T
from monai.data import Dataset, CacheDataset, DataLoader
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.transforms import MapTransform

set_determinism(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MONAI {monai.__version__} | PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | Total memory: {total_mem:.1f} GB')

# Disk space check
usage = shutil.disk_usage('/kaggle/working')
free_gb = usage.free / 1e9
print(f'Disk free: {free_gb:.1f} GB (need ~2GB for checkpoints)')
if free_gb < 5:
    print('WARNING: low disk space!')


In [ ]:
# BraTS 2024 Post-Treatment ground truth labels:
#   0 = Background
#   1 = NETC (Non-Enhancing Tumor Core — necrosis/cysts)
#   2 = SNFH (Surrounding Non-enhancing FLAIR Hyperintensity — edema)
#   3 = ET   (Enhancing Tissue — active tumor)
#   4 = RC   (Resection Cavity — fluid/blood/air)
#
# Evaluation sub-regions (from BraTS 2024 challenge spec):
#   ET = label 3
#   TC = ET + NETC       = labels 3 + 1
#   WT = ET + SNFH + NETC = labels 3 + 2 + 1
#   RC is NOT part of WT or TC!
#
# 3-channel output: [WT, TC, ET]

class ConvertToMultiChannelBrats3Chd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT = NETC+SNFH+ET (no RC)
                (img==1)|(img==3),           # TC = NETC+ET
                img==3,                      # ET = Enhancing Tissue
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

print('BraTS 2024 Label mapping: WT=1+2+3 | TC=1+3 | ET=3')
print('  1=NETC  2=SNFH  3=ET  4=RC(excluded)')


In [ ]:
SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / nii_gz.parent.name / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
for ds_dir in DATA_ROOT.iterdir():
    if not ds_dir.is_dir(): continue
    if list(ds_dir.rglob('*.nii_gz')):
        n = setup_nii_gz_symlinks(ds_dir)
        if n: print(f'  Created {n} symlinks in {ds_dir.name}')

NIFTI_ROOT = None
for search_root in [SYMLINK_DIR, DATA_ROOT]:
    if not search_root.exists(): continue
    for c in search_root.rglob('BraTS-GLI-*'):
        if c.is_dir():
            NIFTI_ROOT = c.parent
            break
    if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    raise RuntimeError('No BraTS-GLI-* folders found - check dataset attachments')
print(f'NIFTI_ROOT: {NIFTI_ROOT}')

split_map, scan_meta = {}, {}
for f in DATA_ROOT.rglob('scan_index.json'):
    si = json.load(open(f))
    for s in si.get('training_scans', []):
        pid, sid = s['patient_id'], s['scan_id']
        split = s.get('split')
        if split: split_map[pid] = split
        scan_meta[sid] = {'patient_id': pid, 'timepoint': s.get('timepoint','100'), 'split': split}
    print(f'  Split metadata: {len(split_map)} patients from scan_index.json')
    break

all_dirs = sorted([d for d in NIFTI_ROOT.iterdir() if d.is_dir() and 'BraTS-GLI' in d.name])
training_scans = []
for d in all_dirs:
    files = {m: list(d.glob(f'*-{m}*')) for m in ['t1n','t1c','t2w','t2f']}
    seg   = list(d.glob('*-seg*'))
    if not (all(files[m] for m in files) and seg): continue
    name = d.name; pid = name.rsplit('-',1)[0]; tp = name.rsplit('-',1)[1] if '-' in name else '100'
    split = scan_meta.get(name, {}).get('split') or split_map.get(pid)
    training_scans.append({
        'scan_id': name, 'patient_id': pid, 'timepoint': tp,
        't1n': str(files['t1n'][0]), 't1c': str(files['t1c'][0]),
        't2w': str(files['t2w'][0]), 't2f': str(files['t2f'][0]),
        'seg': str(seg[0]), 'split': split,
    })
print(f'Total scans: {len(training_scans)} from {NIFTI_ROOT}')

if any(s['split'] for s in training_scans):
    train_scans = [s for s in training_scans if s.get('split')=='train']
    val_scans   = [s for s in training_scans if s.get('split')=='val']
    kt = {s['patient_id'] for s in train_scans}; kv = {s['patient_id'] for s in val_scans}
    for s in training_scans:
        if s.get('split'): continue
        if s['patient_id'] in kt: train_scans.append(s)
        elif s['patient_id'] in kv: val_scans.append(s)
        else: train_scans.append(s)
else:
    from collections import defaultdict
    pts = defaultdict(list)
    for s in training_scans: pts[s['patient_id']].append(s)
    pids = sorted(pts.keys()); n80 = int(0.8*len(pids))
    tp_set = set(pids[:n80]); vp_set = set(pids[n80:])
    train_scans = [s for s in training_scans if s['patient_id'] in tp_set]
    val_scans   = [s for s in training_scans if s['patient_id'] in vp_set]

print(f'Train: {len(train_scans)} scans | Val: {len(val_scans)} scans')


In [ ]:
patch = [128, 128, 128]
train_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.RandFlipd(keys=['image','label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[2], prob=0.5),
    T.RandScaleIntensityd(keys='image', factors=0.1, prob=0.3),
    T.RandShiftIntensityd(keys='image', offsets=0.1, prob=0.3),
    T.SpatialPadd(keys=['image','label'], spatial_size=patch),
    T.RandCropByPosNegLabeld(keys=['image','label'], label_key='label',
        spatial_size=patch, pos=2, neg=1, num_samples=2, image_key='image', image_threshold=0),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print('Transforms ready (3-channel: WT/TC/ET)')


In [ ]:
import nibabel as nib

def is_valid_gzip(path):
    # Check gzip magic bytes (31, 139) and minimum size
    try:
        with open(str(path), 'rb') as f:
            h = f.read(10)
            if len(h) < 10 or h[0] != 31 or h[1] != 139:
                return False
            f.seek(0, 2)
            return f.tell() > 1024
    except Exception:
        return False

def validate_scan(s):
    try:
        for key in ['t1n','t1c','t2w','t2f','seg']:
            if not is_valid_gzip(s[key]): return False
            _ = nib.load(s[key]).shape
        return True
    except Exception:
        return False

def build_dicts(scan_list):
    dicts, bad = [], []
    for s in scan_list:
        if not validate_scan(s):
            bad.append(s['scan_id']); continue
        dicts.append({
            'image': [s['t1n'],s['t1c'],s['t2w'],s['t2f']],
            'label': s['seg'],
            'patient_id': s['patient_id'],
            'timepoint':  s['timepoint'],
        })
    if bad: print(f'  Skipped {len(bad)} corrupted: {bad[:3]}{"..." if len(bad)>3 else ""}')
    return dicts

print('Validating scans (gzip header check)...')
train_dicts = build_dicts(train_scans)
val_dicts   = build_dicts(val_scans)
print(f'Train dicts: {len(train_dicts)} | Val dicts: {len(val_dicts)}')
if train_dicts:
    print(f'  Sample: {train_dicts[0]["image"][0]}')
    print(f'  Exists: {Path(train_dicts[0]["image"][0]).exists()}')


In [ ]:
print('='*55)
print('  Loading nnUNet v2 pretrained model')
print('  Target: WT=0.9005  TC=0.8673  ET=0.8509 (BraTS 2021)')
print('='*55)

# ── Step 1: Find checkpoint + plans ──
ckpt_path  = None
plans_path = None
for p in Path('/kaggle/input').rglob('checkpoint_final.pth'):
    ckpt_path = p; break
for fname in ['plans.json', 'nnUNetPlans.json']:  # handles both naming conventions
    for p in Path('/kaggle/input').rglob(fname):
        plans_path = p; break
    if plans_path: break

print(f'Checkpoint: {ckpt_path}')
print(f'Plans:      {plans_path}')

model = None

def safe_torch_load(path):
    # PyTorch 2.6+ blocks numpy in checkpoints with weights_only=True
    # Try weights_only=True first, then add numpy safe globals, then fall back
    import numpy
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        pass
    try:
        safe = [numpy.core.multiarray.scalar, numpy.dtype, numpy.ndarray]
        with torch.serialization.safe_globals(safe):
            return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        pass
    # Final fallback - weights_only=False (checkpoint is our own trusted upload)
    return torch.load(path, map_location='cpu', weights_only=False)

# -- Step 2: Build PlainConvUNet directly from plans.json (bypasses get_network_from_plans API changes) --
if ckpt_path and plans_path:
    try:
        import json as _j, torch.nn as nn

        plans = _j.load(open(plans_path))
        cfg   = plans['configurations']['3d_fullres']

        # Read exact architecture from plans
        arch_class = cfg.get('UNet_class_name', 'PlainConvUNet')
        n_stages   = len(cfg['conv_kernel_sizes'])
        base_f     = cfg.get('UNet_base_num_features', 32)
        max_f      = cfg.get('unet_max_num_features', 320)
        features   = [min(base_f * (2**i), max_f) for i in range(n_stages)]
        strides    = cfg['pool_op_kernel_sizes']          # already includes [1,1,1] as first
        kernels    = cfg['conv_kernel_sizes']
        n_enc      = cfg.get('n_conv_per_stage_encoder', [2]*n_stages)
        n_dec      = cfg.get('n_conv_per_stage_decoder', [2]*(n_stages-1))
        print(f'  {arch_class} | {n_stages} stages | features: {features}')
        print(f'  strides: {strides}')

        # Import PlainConvUNet - try multiple paths (changed across nnunetv2 versions)
        PlainConvUNet = None
        for imp in [
            ('dynamic_network_architectures.architectures.unet', 'PlainConvUNet'),
            ('nnunetv2.architectures.neural_network',            'PlainConvUNet'),
            ('nnunetv2.nets.UNet',                               'PlainConvUNet'),
        ]:
            try:
                mod = __import__(imp[0], fromlist=[imp[1]])
                PlainConvUNet = getattr(mod, imp[1])
                print(f'  Imported from {imp[0]}')
                break
            except Exception:
                continue

        if PlainConvUNet is None:
            raise ImportError('Could not import PlainConvUNet from any known path')

        model = PlainConvUNet(
            input_channels          = 4,         # T1N, T1C, T2W, T2F
            n_stages                = n_stages,
            features_per_stage      = features,
            conv_op                 = nn.Conv3d,
            kernel_sizes            = kernels,
            strides                 = strides,
            n_conv_per_stage        = n_enc,
            num_classes             = 3,          # WT, TC, ET
            n_conv_per_stage_decoder= n_dec,
            conv_bias               = False,
            norm_op                 = nn.InstanceNorm3d,
            norm_op_kwargs          = {'eps': 1e-05, 'affine': True},
            dropout_op              = None,
            dropout_op_kwargs       = None,
            nonlin                  = nn.LeakyReLU,
            nonlin_kwargs           = {'inplace': True},
            deep_supervision        = False,
        )

        # Load pretrained weights (safe_torch_load handles PyTorch 2.6 weights_only change)
        ckpt   = safe_torch_load(ckpt_path)
        state  = ckpt.get('network_weights', ckpt.get('state_dict', ckpt))
        own    = model.state_dict()
        compat = {k: v for k,v in state.items() if k in own and own[k].shape == v.shape}
        model.load_state_dict({**own, **compat}, strict=False)
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  Pretrained: {len(compat)}/{len(own)} layers loaded | {n_params/1e6:.1f}M params')
        if len(compat) < len(own)//2:
            print('  WARNING: <50% layers matched - weight keys may differ')

    except Exception as e:
        print(f'  PlainConvUNet build failed: {e}')
        print('  Falling back to MONAI DynUNet ...')
        model = None

# -- Step 3: Fallback -- MONAI DynUNet (training from scratch) --
if model is None:
    from monai.networks.nets import DynUNet
    kernels  = [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]]
    strides  = [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]]
    model = DynUNet(
        spatial_dims=3, in_channels=4, out_channels=3,
        kernel_size=kernels, strides=strides,
        upsample_kernel_size=strides[1:],
        norm_name='INSTANCE', deep_supervision=False, res_block=True,
    )
    print(f'  DynUNet: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params (training from scratch)')

model = model.to(device)
print(f'Model on {device}')


In [ ]:
import math, time, gc
from torch.cuda.amp import GradScaler, autocast

CKPT_DIR    = OUTPUT_ROOT / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH   = CKPT_DIR / 'nnunet_best.pth'
LATEST_PATH = CKPT_DIR / 'nnunet_latest.pth'

# ── Recover checkpoints from previous notebook output (attached as input) ──
if not BEST_PATH.exists() or not LATEST_PATH.exists():
    for src_f in sorted(Path('/kaggle/input').rglob('nnunet_best.pth')):
        if not BEST_PATH.exists():
            import shutil; shutil.copy2(src_f, BEST_PATH)
            print(f'  Recovered BEST checkpoint from {src_f}')
        break
    for src_f in sorted(Path('/kaggle/input').rglob('nnunet_latest.pth')):
        if not LATEST_PATH.exists():
            import shutil; shutil.copy2(src_f, LATEST_PATH)
            print(f'  Recovered LATEST checkpoint from {src_f}')
        break
    if BEST_PATH.exists():
        print(f'  ✅ Checkpoints recovered — training will be skipped')

def get_lr(ep, total, base=1e-4):
    warm = 5
    if ep < warm: return base * (ep+1) / warm
    return base * 0.5 * (1 + math.cos(math.pi * (ep-warm) / max(total-warm, 1)))

def safe_loader_iter(loader):
    # MUST use iter/next  - 'for batch in loader' re-raises worker exceptions
    # BEFORE entering the loop body, bypassing any inner try/except
    it = iter(loader)
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream','worker','corrupt']
    while True:
        try:
            yield next(it)
        except StopIteration:
            return
        except Exception as e:
            if any(k in str(e) for k in SKIP):
                continue   # skip bad file, keep training
            raise

def train_model(model, lr=1e-4, epochs=30, patience=10, val_interval=4):
    start_ep, best_dice, mlog = 0, 0.0, {'dice':[],'per_region':[],'loss':[]}
    if LATEST_PATH.exists():
        lc = torch.load(LATEST_PATH, map_location='cpu')
        model.load_state_dict(lc['model'])
        start_ep  = lc.get('epoch',0) + 1
        best_dice = lc.get('best_dice', 0)
        mlog      = lc.get('metrics', mlog)
        print(f'Resumed from epoch {start_ep-1}, best_dice={best_dice:.4f}')
        if start_ep >= epochs:
            return model, best_dice, mlog

    loss_fn     = DiceLoss(to_onehot_y=False, sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scaler      = GradScaler()
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    no_improve  = 0
    t0          = time.time()

    # Dataset strategy:
    # Train: CacheDataset 5% cache (~66 scans in RAM) — stable API across all MONAI versions
    # Val:   Plain Dataset ONLY — no disk cache (PersistentDataset = ~60GB -> kills Kaggle disk)
    try:
        from monai.data import CacheDataset
        train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=0.05, num_workers=4)
        print('Using CacheDataset (5% RAM cache)')
    except Exception:
        train_ds = Dataset(train_dicts, train_transforms)
        print('Using plain Dataset (CacheDataset unavailable)')
    val_ds = Dataset(val_dicts, val_transforms)  # plain disk read - no storage bloat

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')
    print(f'Expected ~20 min/epoch x {epochs} epochs ~ {epochs*20/60:.1f}h total')

    for ep in range(start_ep, epochs):
        model.train()
        cur_lr = get_lr(ep, epochs, lr)
        for pg in optimizer.param_groups: pg['lr'] = cur_lr

        ep_loss, n_ok, n_bad = 0.0, 0, 0
        for batch in safe_loader_iter(train_loader):
            try:
                imgs = batch['image'].to(device)
                lbls = batch['label'].to(device)
                optimizer.zero_grad()
                with autocast():
                    loss = loss_fn(model(imgs), lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
                ep_loss += loss.item(); n_ok += 1
            except Exception:
                n_bad += 1
        avg_loss = ep_loss / max(n_ok, 1)
        bad_str  = f' | skipped {n_bad}' if n_bad else ''
        mlog['loss'].append(avg_loss)

        if (ep+1) % val_interval == 0 or ep == epochs-1:
            model.eval(); dice_metric.reset()
            with torch.no_grad():
                for vb in safe_loader_iter(val_loader):
                    try:
                        vo = sliding_window_inference(vb['image'].to(device), PATCH, 2, model, overlap=0.25)
                        dice_metric((torch.sigmoid(vo)>0.5).float(), vb['label'].to(device))
                    except Exception:
                        pass
            dv = dice_metric.aggregate(); md = dv.mean().item()
            pr = [round(dv[i].item(),4) for i in range(3)]  # WT, TC, ET
            mlog['dice'].append(md); mlog['per_region'].append(pr)
            tag = ' NEW BEST' if md > best_dice else ''
            print(f'Ep {ep:3d} | L={avg_loss:.4f} | Dice={md:.4f} WT={pr[0]:.3f} TC={pr[1]:.3f} ET={pr[2]:.3f} | {(time.time()-t0)/60:.1f}m{tag}{bad_str}')
            if md > best_dice:
                best_dice = md; no_improve = 0
                torch.save({'model': model.state_dict(), 'epoch': ep, 'best_dice': best_dice}, BEST_PATH)
            else:
                no_improve += val_interval
        else:
            print(f'Ep {ep:3d} | L={avg_loss:.4f} | LR={cur_lr:.2e} | {(time.time()-t0)/60:.1f}m{bad_str}')

        torch.save({'model': model.state_dict(), 'epoch': ep,
                    'best_dice': best_dice, 'metrics': mlog}, LATEST_PATH)
        if no_improve >= patience:
            print(f'Early stop at ep {ep}'); break

    print(f'Done. Best Mean Dice = {best_dice:.4f}')
    print(f'  Target: WT=0.900 TC=0.867 ET=0.851 (BraTS 2021 benchmark)')
    return model, best_dice, mlog

print('Fine-tuning nnUNet on BraTS 2024 Post-Treatment...')
# 30 epochs x ~20min = ~10h -- leaves ~2h for embedding extraction + visualization
model, best_dice, metrics = train_model(model, lr=1e-4, epochs=28, patience=10)


In [ ]:
import gc, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from monai.metrics import DiceMetric
import random
import numpy as np
import math
import time

fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(exist_ok=True)
REGION_COLORS = {'WT': ('red','lightcoral'), 'TC': ('green','lightgreen'), 'ET': ('blue','lightskyblue')}

# ── Load BEST checkpoint before visualization ──
if BEST_PATH.exists():
    best_ckpt = torch.load(BEST_PATH, map_location='cpu')
    model.load_state_dict(best_ckpt['model'])
    print(f'  Loaded BEST checkpoint → epoch {best_ckpt["epoch"]} | Dice {best_ckpt["best_dice"]:.4f}')
else:
    print('  BEST checkpoint not found — using current weights')
gc.collect(); torch.cuda.empty_cache()
print(f'  VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.1f} GB')

# ══════════════════════════════════════════════
# 3D Voxel-Scatter Visualization
# ══════════════════════════════════════════════
def visualize_3d_predictions(model, n_samples=5):
    """5 random val patients, sw_batch_size=1, overlap=0.25 to save VRAM."""
    model.eval()
    vis_dicts = random.sample(val_dicts, min(n_samples, len(val_dicts)))
    print(f'  Visualising {len(vis_dicts)} random val patients:')
    for s in vis_dicts: print(f'    {s["patient_id"]} tp={s["timepoint"]}')
    dice_metric = DiceMetric(include_background=True, reduction='none')
    vis_ds     = Dataset(vis_dicts, val_transforms)
    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    for idx, batch in enumerate(vis_loader):
        try:
            gc.collect(); torch.cuda.empty_cache()  # flush before each sample
            vi = batch['image'].to(device)
            vl = batch['label'].to(device)
            with torch.no_grad():
                vo = sliding_window_inference(vi, PATCH, 1, model, overlap=0.25)
            pred_bin = (torch.sigmoid(vo) > 0.5).float()
            # Compute Dice BEFORE deleting tensors
            dice_metric.reset(); dice_metric(pred_bin, vl)
            dv  = dice_metric.aggregate()[0].cpu().numpy()
            pid = batch.get('patient_id', ['?'])[0]
            def fmt(v): return 'NaN(empty GT)' if np.isnan(v) else f'{v:.3f}'
            dstr = '  '.join(f'{r}={fmt(dv[j])}' for j,r in enumerate(REGIONS))
            mean_d = float(np.nanmean(dv))
            title_str = f'Sample {idx} | {dstr}  Mean={mean_d:.3f}  {pid}'
            # Move to CPU THEN free GPU
            pred_np = pred_bin.squeeze(0).cpu().numpy()
            gt_np   = vl.squeeze(0).cpu().numpy()
            del vi, vl, vo, pred_bin; gc.collect(); torch.cuda.empty_cache()
            # Plot
            STEP = 3
            fig = plt.figure(figsize=(18, 7))
            fig.suptitle(title_str, fontsize=10, y=1.01)
            for col, (arr, col_title) in enumerate([(gt_np, 'Ground Truth'), (pred_np, 'nnUNet Prediction')]):
                ax = fig.add_subplot(1, 2, col+1, projection='3d')
                ax.set_title(col_title, fontsize=11, pad=8)
                for r_idx, rname in enumerate(REGIONS):
                    mask = arr[r_idx]; coords = (mask > 0.5).nonzero()
                    if len(coords[0]) == 0: continue
                    xs, ys, zs = coords[0][::STEP], coords[1][::STEP], coords[2][::STEP]
                    color = REGION_COLORS[rname][0] if col==0 else REGION_COLORS[rname][1]
                    ax.scatter(xs, ys, zs, c=color, alpha=0.25, s=0.8, label=rname)
                ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
                ax.tick_params(labelsize=7); ax.legend(markerscale=6, loc='upper left', fontsize=8)
            plt.tight_layout()
            out_png = fig_dir / f'nnunet_3d_sample{idx}.png'
            plt.savefig(out_png, dpi=120, bbox_inches='tight'); plt.close()
            print(f'  Saved: {out_png.name} | {dstr}')
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            print(f'  OOM sample {idx} — skipping')
        except Exception as e:
            print(f'  Skipped sample {idx}: {e}')

# ══════════════════════════════════════════════
# 2D Overlay Visualization
# ══════════════════════════════════════════════
def visualize_2d_overlays(model, n_samples=5):
    """5 random val patients, 2D T1c slice overlay."""
    model.eval()
    vis_sample = random.sample(val_dicts, min(n_samples, len(val_dicts)))
    vis_ds = Dataset(vis_sample, val_transforms)
    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    for idx, batch in enumerate(vis_loader):
        try:
            gc.collect(); torch.cuda.empty_cache()
            img = batch['image'].to(device); lbl = batch['label'].to(device)
            with torch.no_grad():
                vo = sliding_window_inference(img, PATCH, 1, model, overlap=0.25)
            pred_bin = (torch.sigmoid(vo) > 0.5).float().squeeze(0).cpu().numpy()
            t1c_np   = img[0, 1].cpu().numpy()
            del img, lbl, vo; gc.collect(); torch.cuda.empty_cache()
            wt_mask = pred_bin[0]
            if wt_mask.sum() == 0:
                print(f'  Sample {idx}: no WT voxels — skip'); continue
            z_slice   = int(np.argmax(wt_mask.sum(axis=(0,1))))
            t1c_slice = t1c_np[:, :, z_slice]
            edema     = np.logical_and(pred_bin[0,:,:,z_slice]>0, pred_bin[1,:,:,z_slice]==0)
            necrosis  = np.logical_and(pred_bin[1,:,:,z_slice]>0, pred_bin[2,:,:,z_slice]==0)
            enhancing = pred_bin[2,:,:,z_slice] > 0
            mask_rgb  = np.zeros((*t1c_slice.shape, 3))
            mask_rgb[edema]     = [0,1,0]  # Green = SNFH/Edema
            mask_rgb[enhancing] = [0,0,1]  # Blue  = ET
            mask_rgb[necrosis]  = [1,0,0]  # Red   = NETC
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            pid = batch.get('patient_id', ['?'])[0]
            axes[0].set_title(f'T1c — z={z_slice} | {pid}')
            axes[0].imshow(t1c_slice, cmap='gray'); axes[0].axis('off')
            axes[1].set_title('Overlay  (G:SNFH  B:ET  R:NETC)')
            axes[1].imshow(t1c_slice, cmap='gray')
            axes[1].imshow(mask_rgb, alpha=0.45); axes[1].axis('off')
            plt.tight_layout()
            out_path = fig_dir / f'nnunet_2d_sample{idx}.png'
            plt.savefig(out_path, dpi=120, bbox_inches='tight'); plt.close()
            print(f'  Saved: {out_path.name} | patient={pid}')
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            print(f'  OOM sample {idx} — skipping')
        except Exception as e:
            print(f'  Skip 2D sample {idx}: {e}')

print('Generating 3D voxel-scatter visualizations (5 random patients)...')
visualize_3d_predictions(model, n_samples=5)

print('Generating 2D overlay visualizations (5 random patients)...')
visualize_2d_overlays(model, n_samples=5)
print(f'Figures saved to: {fig_dir}')


In [ ]:
import torch.nn.functional as F
import gc, time, random
import numpy as np

# ═══════════ CNN (nnUNet) Embedding Extraction v2 ═════════════════
#
# Aligned with SwinUNETR v2 strategy (Design Decisions D7/D8):
#   - Hook mid encoder stage (~16×16×16 resolution, C channels)
#   - ROI-crop to WT bounding box + 1-cell padding
#   - Octant spatial pooling: 2×2×2 adaptive avg → 8 × C = 8C dims
#   - Mask-weighted pooling: WT/TC/ET within ROI → 3 × C dims
#   - Volumetric morphology: 9-D (log-volumes + presence + ratios)
#   - Total: 8C + 3C + 9 = 11C + 9 dims
#
# For complete resection (no WT):
#   → Octant + region = zero vectors, vol: has_wt=0, etc.
#   → Tumor absence IS the evolution signal
#
# Additionally saves spatial tokens for Phase 4 (RadFM) / Phase 5 (TaDiff)
# ═══════════════════════════════════════════════════════════════════

def _is_corrupt_file_error(exc):
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream',
            'corrupt','truncat','LoadImaged','applying transform']
    e = exc
    while e is not None:
        if any(k in (type(e).__name__+' '+str(e)) for k in SKIP): return True
        e = e.__cause__ or e.__context__
    return False

def safe_emb_iter(loader):
    it, n_skip = iter(loader), 0
    while True:
        try: yield next(it)
        except StopIteration:
            if n_skip: print(f'  Skipped {n_skip} corrupt files total')
            return
        except Exception as e:
            if _is_corrupt_file_error(e): n_skip += 1; continue
            raise

def get_wt_bbox(lbl_3ch, min_size=2):
    """Find bounding box of WT (channel 0) in 16×16×16 label space."""
    wt = lbl_3ch[0]  # (D, H, W)
    coords = (wt > 0.5).nonzero(as_tuple=True)
    if len(coords[0]) < min_size:
        return None
    z0, z1 = int(coords[0].min()), int(coords[0].max()) + 1
    y0, y1 = int(coords[1].min()), int(coords[1].max()) + 1
    x0, x1 = int(coords[2].min()), int(coords[2].max()) + 1
    # Pad by 1 cell, clamp to valid range
    D, H, W = wt.shape
    z0, z1 = max(z0-1, 0), min(z1+1, D)
    y0, y1 = max(y0-1, 0), min(y1+1, H)
    x0, x1 = max(x0-1, 0), min(x1+1, W)
    # Ensure each dim >= 2 for adaptive_avg_pool3d(2,2,2)
    if z1-z0 < 2: z1 = min(z0+2, D)
    if y1-y0 < 2: y1 = min(y0+2, H)
    if x1-x0 < 2: x1 = min(x0+2, W)
    return (z0, z1, y0, y1, x0, x1)

def extract_embeddings(model):
    model.eval()
    _feats = {}; hooks = []

    # ── Hook the MID encoder stage (16×16×16 resolution) ──
    # nnUNet PlainConvUNet: 6 stages, features [32, 64, 128, 256, 320, 320]
    # strides: [[1,1,1], [2,2,2], [2,2,2], [2,2,2], [2,2,2], [2,2,1]]
    # Stage 0: 128³ → 128³ (32ch)
    # Stage 1: 128³ → 64³  (64ch)
    # Stage 2: 64³  → 32³  (128ch)
    # Stage 3: 32³  → 16³  (256ch)   ← we hook THIS (comparable to SwinUNETR layers2)
    # Stage 4: 16³  → 8³   (320ch)
    # Stage 5: 8³   → 4³   (320ch)

    if hasattr(model, 'encoder') and hasattr(model.encoder, 'stages'):
        stages = list(model.encoder.stages)
        n = len(stages)
        # Target stage 3 (16³, 256ch) — matches SwinUNETR layers2 resolution
        # Fallback to n-3 if architecture differs
        target_idx = min(3, n - 1)
        def _hook(m, inp, out):
            feat = out[-1] if isinstance(out, (list, tuple)) else out
            _feats['mid'] = feat.detach()
        hooks.append(stages[target_idx].register_forward_hook(_hook))
        print(f'  Hook: encoder.stages[{target_idx}] ({type(stages[target_idx]).__name__})')
    else:
        target = model.encoder if hasattr(model, 'encoder') else model
        def _hook(m, inp, out):
            feat = out[-1] if isinstance(out, (list, tuple)) else out
            _feats['mid'] = feat.detach()
        hooks.append(target.register_forward_hook(_hook))
        print(f'  Hook fallback: {type(target).__name__}')

    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(exist_ok=True)
    embs, ids, tps = [], [], []
    spatial_tokens_list = []  # Phase 4/5: raw spatial features before pooling
    bboxes_list = []          # ROI bounding boxes

    all_dicts = train_dicts + val_dicts
    ds     = Dataset(all_dicts, val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    total  = len(all_dicts)
    n_skip = 0; n_empty = 0
    t_start = time.time()

    print(f'  Extraction v2: {total} scans')
    print(f'  {"─"*60}')

    with torch.no_grad():
        for idx, batch in enumerate(safe_emb_iter(loader)):
            pid = batch['patient_id'][0]
            tp  = batch['timepoint'][0]
            try:
                img = batch['image'].to(device)
                lbl = batch['label'].to(device)
                img_p = F.interpolate(img, PATCH, mode='trilinear', align_corners=False)
                lbl_p = F.interpolate(lbl.float(), PATCH, mode='nearest')
                _feats.clear()
                _ = model(img_p)

                if 'mid' not in _feats:
                    print(f'  [{idx+1:4d}/{total}] WARN no hook → skip {pid}')
                    n_skip += 1; continue

                feat = _feats['mid']  # (1, C, D, H, W) e.g. (1, 256, 16, 16, 16)
                C = feat.shape[1]

                if idx == 0:
                    print(f'  Feature map: {tuple(feat.shape)} → C={C}')

                # Compute volumes from FULL-res labels
                wt_vol = float(lbl_p[0, 0].sum().item())
                tc_vol = float(lbl_p[0, 1].sum().item())
                et_vol = float(lbl_p[0, 2].sum().item())

                # Downsample labels to feature map resolution
                lbl_feat = F.adaptive_avg_pool3d(lbl_p, feat.shape[2:])

                # ── Component 1: Octant Spatial Pooling (8 × C) ──
                feat_crop = None  # initialize so del is always safe
                bbox = get_wt_bbox(lbl_feat[0], min_size=2)

                if bbox is not None:
                    z0, z1, y0, y1, x0, x1 = bbox
                    feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]  # (1, C, dz, dy, dx)
                    # Octant pool: 2×2×2 adaptive avg → 8 spatial cells × C features
                    oct_pooled = F.adaptive_avg_pool3d(feat_crop, (2, 2, 2))  # (1, C, 2, 2, 2)
                    oct_vec = oct_pooled[0].reshape(C, 8).T.reshape(-1)       # (8C,)
                    # Mask-weighted pool: WT/TC/ET within ROI
                    lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                    feat_flat = feat_crop[0].reshape(C, -1)  # (C, N)
                    region_vecs = []
                    for ch in range(3):  # WT, TC, ET
                        mask = lbl_crop[0, ch].reshape(-1)
                        vol_soft = float(mask.sum().item())
                        if vol_soft > 0.01:
                            rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                        else:
                            rvec = torch.zeros(C, device=device)
                        region_vecs.append(rvec)
                    roi_size = f'{z1-z0}×{y1-y0}×{x1-x0}'
                    # Capture spatial tokens BEFORE del feat_crop
                    sp_tok = feat_crop[0].reshape(C, -1).T.cpu().numpy()  # (N_tok, C)
                    bbox_entry = list(bbox)
                else:
                    # Complete resection — no tumor ROI
                    oct_vec = torch.zeros(8 * C, device=device)
                    region_vecs = [torch.zeros(C, device=device) for _ in range(3)]
                    roi_size = 'empty'
                    n_empty += 1
                    sp_tok = np.zeros((1, C), dtype=np.float32)
                    bbox_entry = [0, 0, 0, 0, 0, 0]

                # ── Component 3: Volumetric morphology (9-D) ──
                log_wt = np.log1p(wt_vol)
                log_tc = np.log1p(tc_vol)
                log_et = np.log1p(et_vol)
                has_wt = 1.0 if wt_vol > 10 else 0.0
                has_tc = 1.0 if tc_vol > 10 else 0.0
                has_et = 1.0 if et_vol > 10 else 0.0
                tc_wt = tc_vol / (wt_vol + 1e-6)
                et_wt = et_vol / (wt_vol + 1e-6)
                et_tc = et_vol / (tc_vol + 1e-6)
                vol_feat = torch.tensor(
                    [log_wt, log_tc, log_et, has_wt, has_tc, has_et,
                     tc_wt, et_wt, et_tc], dtype=torch.float32)

                # ── Concatenate: [octant(8C) + region(3C) + vol(9)] ──
                emb = torch.cat([oct_vec.cpu()] + [v.cpu() for v in region_vecs]
                                + [vol_feat]).numpy()

                # Cleanup GPU tensors
                del img, lbl, img_p, lbl_p, feat
                if feat_crop is not None:
                    del feat_crop
                    feat_crop = None

                # Append spatial tokens + embeddings
                spatial_tokens_list.append(sp_tok)
                bboxes_list.append(bbox_entry)
                embs.append(emb); ids.append(pid); tps.append(tp)

                if idx == 0:
                    print(f'  Embedding: octant={8*C} + region={3*C} + vol=9 = {emb.shape[0]}-D')
                    print(f'  {"─"*60}')

                if (idx+1) % 50 == 0 or idx == 0:
                    elapsed   = time.time() - t_start
                    rate      = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total - idx - 1) / max(rate, 1e-6)
                    print(
                        f'  [{idx+1:4d}/{total}] {pid[:30]:<30}'
                        f'  tp={tp}'
                        f'  WT={wt_vol:6.0f}v TC={tc_vol:6.0f}v ET={et_vol:6.0f}v'
                        f'  ROI={roi_size}'
                        f'  | {rate:.1f}/s ETA {remaining/60:.1f}m'
                    )
                elif (idx+1) % 10 == 0:
                    elapsed   = time.time() - t_start
                    rate      = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total - idx - 1) / max(rate, 1e-6)
                    print(f'  [{idx+1:4d}/{total}]  done={len(embs)} | {rate:.1f}/s ETA {remaining/60:.1f}m')

            except Exception as e:
                print(f'  [{idx+1:4d}/{total}] ERROR {pid}: {str(e)[:70]}')
                n_skip += 1; continue

    for hk in hooks:
        try: hk.remove()
        except: pass

    elapsed_total = time.time() - t_start
    print(f'  {"─"*60}')
    print(f'  Done: {len(embs)}/{total} in {elapsed_total/60:.1f} min')
    print(f'  Skipped: {n_skip} | Empty ROI (complete resection): {n_empty}')

    if not embs: raise RuntimeError('No embeddings extracted.')
    arr = np.array(embs)
    D = arr.shape[1]
    out = emb_dir / 'cnn_nnunet_embeddings.npz'
    np.savez_compressed(out, embeddings=arr,
                        patient_ids=np.array(ids), timepoints=np.array(tps))
    print(f'  Saved: {out}  shape={arr.shape} ({arr.nbytes/1e6:.1f} MB)')

    # ── Save spatial tokens for Phase 4/5 ──
    if spatial_tokens_list:
        max_tokens = max(t.shape[0] for t in spatial_tokens_list)
        C_tok = spatial_tokens_list[0].shape[1]
        padded = np.zeros((len(spatial_tokens_list), max_tokens, C_tok), dtype=np.float32)
        n_tokens = np.zeros(len(spatial_tokens_list), dtype=np.int32)
        for j, tok in enumerate(spatial_tokens_list):
            padded[j, :tok.shape[0], :] = tok
            n_tokens[j] = tok.shape[0]
        tok_out = emb_dir / 'cnn_spatial_tokens.npz'
        np.savez_compressed(tok_out,
            spatial_tokens=padded,
            token_counts=n_tokens,
            patient_ids=np.array(ids),
            timepoints=np.array(tps),
            bboxes=np.array(bboxes_list)
        )
        print(f'  Spatial tokens: {tok_out}')
        print(f'    Shape: {padded.shape} ({padded.nbytes/1e6:.1f} MB)')
        print(f'    Max tokens: {max_tokens}  Mean: {n_tokens.mean():.0f}')

    # ── Save tumor_volumes.csv for eval notebook ──
    import pandas as pd
    vol_rows = []
    for j, (pid, tp, emb) in enumerate(zip(ids, tps, embs)):
        v = emb[-9:]  # last 9 dims = volumetric features
        vol_rows.append({
            'patient_id': pid, 'timepoint': tp,
            'wt_vol': float(np.expm1(v[0])),
            'tc_vol': float(np.expm1(v[1])),
            'et_vol': float(np.expm1(v[2])),
            'has_wt': float(v[3]), 'has_tc': float(v[4]), 'has_et': float(v[5]),
            'tc_wt_ratio': float(v[6]), 'et_wt_ratio': float(v[7]),
            'et_tc_ratio': float(v[8]),
        })
    vol_df = pd.DataFrame(vol_rows)
    csv_out = emb_dir / 'tumor_volumes.csv'
    vol_df.to_csv(csv_out, index=False)
    print(f'  Volumes CSV: {csv_out}  ({len(vol_df)} rows)')
    C_val = D - 9  # total dim minus vol features
    n_oct = (C_val * 8) // 11
    n_reg = (C_val * 3) // 11
    print(f'  Architecture: octant={n_oct}D + region={n_reg}D + vol=9D = {D}D')
    return arr

import gc
print('Cleaning VRAM before embedding extraction...')
if 'optimizer' in globals(): del optimizer
if 'scaler'    in globals(): del scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    print(f'VRAM free: {free_gb:.1f} GB')
else:
    print('Running on CPU')

print('\nExtracting nnUNet embeddings v2 (ROI-crop + octant + volumetric)...')
embeddings = extract_embeddings(model)


In [ ]:
import random

npz_path = OUTPUT_ROOT / 'embeddings' / 'cnn_nnunet_embeddings_v2.npz'
if npz_path.exists():
    data   = np.load(npz_path)
    ids_arr = data['patient_ids']
    emb_arr = data['embeddings']     # (N, D)
    N, D    = emb_arr.shape
    norms   = np.linalg.norm(emb_arr, axis=1)

    # Cosine similarity on random pairs
    emb_norm = emb_arr / (norms[:, None] + 1e-8)
    n_pairs  = min(200, N*(N-1)//2)
    pairs    = random.sample([(i,j) for i in range(N) for j in range(i+1,N)], n_pairs)
    sims     = [float(np.dot(emb_norm[i], emb_norm[j])) for i,j in pairs]
    cos_mean = float(np.mean(sims))
    diverse  = 100.0 * float(np.mean(np.array(sims) < 0.95))
    status   = 'GOOD diversity' if diverse > 20 else 'LOW diversity - check embedding hook'

    print('Embedding Diversity Check')
    print(f'  Scans:    {N}')
    print(f'  Dim:      {D}  (global + 3×region + 9 volumetric)')
    print(f'  Norm:     [{norms.min():.2f}, {norms.max():.2f}]  mean={norms.mean():.2f}')
    print(f'  Cos sim:  mean={cos_mean:.3f} over {n_pairs} random pairs')
    print(f'  Diverse:  {diverse:.1f}% of pairs have cos < 0.95  [{status}]')

    # Quick per-timepoint check
    tps = data['timepoints']
    for tp in sorted(set(tps)):
        idx = [i for i,t in enumerate(tps) if t == tp]
        print(f'  Timepoint {tp}: {len(idx)} scans')
else:
    print(f'Embeddings not found at {npz_path}')
    print('  Run Cell 10 (embedding extraction) first')


In [ ]:
import json as _j
summary = {
    'model': MODEL_NAME, 'best_dice': float(best_dice),
    'regions': REGIONS, 'label': 'BraTS2024: WT=1+2+3(NETC+SNFH+ET), TC=1+3(NETC+ET), ET=3, RC=4(excluded)',
    'train_scans': len(train_dicts), 'val_scans': len(val_dicts),
    'target_brats2021': {'WT': 0.9005, 'TC': 0.8673, 'ET': 0.8509},
}
(OUTPUT_ROOT / 'summary.json').write_text(_j.dumps(summary, indent=2))

print('='*55)
print(f'  nnUNet Fine-Tuning Complete')
print(f'  Best Mean Dice:  {best_dice:.4f}')
print(f'  Regions:         {REGIONS}')
print(f'  Target (BraTS2021): WT=0.900 TC=0.867 ET=0.851')
print(f'  Outputs: {OUTPUT_ROOT}')
print('='*55)
